# Handwritten Digit CNN Comparison

**Author: Feras Elghandour**

Compare a shallow convolutional baseline with a deeper regularized classifier on low-resolution handwritten digits.

Stratified training/validation/test splits, normalization, shallow versus BatchNorm/Dropout CNNs, training curves, confusion matrices, and per-class evaluation.

Read README.md for result scope and DATA.md for input preparation. Run cells in order in a clean environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report
)

print(f"NumPy:      {np.__version__}")
print(f"Pandas:     {pd.__version__}")
print(f"PyTorch:    {torch.__version__}")
print(f"Device:     {'cuda' if torch.cuda.is_available() else 'cpu'}")

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cpu')

In [ ]:
# Load dataset
digits = load_digits()

print(f"Total samples:    {len(digits.data)}")
print(f"Image shape:      {digits.images.shape[1:]} (H x W pixels)")
print(f"Feature vectors:  {digits.data.shape[1]} (flattened 8x8)")
print(f"Classes:          {digits.target_names}")
print(f"Pixel range:      [{digits.data.min():.0f}, {digits.data.max():.0f}]")

# Class distribution
unique, counts = np.unique(digits.target, return_counts=True)
print("\nClass distribution:")
for u, c in zip(unique, counts):
    print(f"  Digit {u}: {c} samples")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 10, figsize=(14, 3.5))
for digit in range(10):
    idx = np.where(digits.target == digit)[0][0]
    axes[0, digit].imshow(digits.images[idx], cmap='gray_r')
    axes[0, digit].set_title(f'Digit {digit}', fontsize=9)
    axes[0, digit].axis('off')
    idx2 = np.where(digits.target == digit)[0][1]
    axes[1, digit].imshow(digits.images[idx2], cmap='gray_r')
    axes[1, digit].axis('off')

fig.suptitle('Figure 0 — Two Representative Samples per Digit Class (8×8 pixels)', fontsize=11)
plt.tight_layout()
plt.show()
print("Each image is an 8×8 grayscale scan (intensity 0–16). Spatial structure is visible despite low resolution.")

In [ ]:
# Reshape: (1797,) → (1797, 1, 8, 8)
X = digits.images.reshape(-1, 1, 8, 8).astype(np.float32)
y = digits.target.astype(np.int64)

# Stratified splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

# Normalize pixel values to [0, 1]
X_train = X_train / 16.0
X_val   = X_val   / 16.0
X_test  = X_test  / 16.0

print(f"Train set:  {X_train.shape[0]} samples")
print(f"Val set:    {X_val.shape[0]} samples")
print(f"Test set:   {X_test.shape[0]} samples")
print(f"Input shape per sample: {X_train.shape[1:]}  (channels x H x W)")

def make_loader(X, y, batch_size=32, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   shuffle=False)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)

print(f"\nTrain batches: {len(train_loader)}  |  Val batches: {len(val_loader)}  |  Test batches: {len(test_loader)}")

In [ ]:
class BaselineCNN(nn.Module):
    """
    Shallow two-block CNN for 10-class digit classification.
    Input:  (B, 1, 8, 8) normalized grayscale images.
    Output: (B, 10) raw logits for CrossEntropyLoss.
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),  # → (B, 16, 8, 8)
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), # → (B, 32, 8, 8)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # → (B, 32, 4, 4)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                 # → (B, 512)
            nn.Linear(32 * 4 * 4, 64),   # → (B, 64)
            nn.ReLU(),
            nn.Linear(64, 10),           # → (B, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Count parameters
model_a = BaselineCNN()
total_params_a = sum(p.numel() for p in model_a.parameters())
print(f"Baseline CNN architecture:")
print(model_a)
print(f"\nTotal trainable parameters: {total_params_a:,}")

In [ ]:
def train_model(model, train_loader, val_loader, epochs=30, lr=0.001, name='Model'):
    """
    Train a PyTorch model and record loss/accuracy history.
    
    Args:
        model:        nn.Module to train
        train_loader: DataLoader for training set
        val_loader:   DataLoader for validation set
        epochs:       number of training epochs
        lr:           Adam learning rate
        name:         display name for logging
    Returns:
        dict with train_loss, val_loss, train_acc, val_acc lists
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            out = model(X_b)
            loss = criterion(out, y_b)
            loss.backward()
            optimizer.step()
            t_loss    += loss.item() * len(y_b)
            t_correct += (out.argmax(1) == y_b).sum().item()
            t_total   += len(y_b)

        # --- Validation ---
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                out = model(X_b)
                v_loss    += criterion(out, y_b).item() * len(y_b)
                v_correct += (out.argmax(1) == y_b).sum().item()
                v_total   += len(y_b)

        history['train_loss'].append(t_loss / t_total)
        history['val_loss'].append(v_loss / v_total)
        history['train_acc'].append(t_correct / t_total)
        history['val_acc'].append(v_correct / v_total)

        if (epoch + 1) % 5 == 0:
            print(f"[{name}] Epoch {epoch+1:2d}/{epochs} | "
                  f"Train Loss: {t_loss/t_total:.4f}  Acc: {t_correct/t_total:.3f} | "
                  f"Val Loss: {v_loss/v_total:.4f}  Acc: {v_correct/v_total:.3f}")

    return history


print("Training Baseline CNN (30 epochs)...")
hist_a = train_model(model_a, train_loader, val_loader, epochs=30, name='Baseline')

In [ ]:
class DeepCNN(nn.Module):
    """
    Deeper CNN with BatchNorm and Dropout regularization.
    Changes vs. BaselineCNN:
      - Three convolutional blocks (vs. two)
      - BatchNorm2d after each Conv2d
      - Dropout(0.4) before the final classifier layer
      - Wider channels: 32 → 64 → 64
    Input:  (B, 1, 8, 8)
    Output: (B, 10) raw logits
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # → (B, 32, 8, 8)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # → (B, 64, 8, 8)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                               # → (B, 64, 4, 4)
            nn.Conv2d(64, 64, kernel_size=3, padding=1),  # → (B, 64, 4, 4)
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                  # → (B, 1024)
            nn.Linear(64 * 4 * 4, 128),   # → (B, 128)
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 10),            # → (B, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model_b = DeepCNN()
total_params_b = sum(p.numel() for p in model_b.parameters())
print(f"Deep CNN architecture:")
print(model_b)
print(f"\nTotal trainable parameters: {total_params_b:,}")
print(f"Parameter increase vs baseline: +{total_params_b - total_params_a:,}")

In [ ]:
print("Training Deep CNN (30 epochs)...")
hist_b = train_model(model_b, train_loader, val_loader, epochs=30, name='DeepCNN')

In [ ]:
def evaluate_model(model, loader, name='Model'):
    """
    Compute accuracy and macro-F1 on a DataLoader.
    Returns: (accuracy, f1, y_true_array, y_pred_array)
    """
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            out = model(X_b.to(device))
            all_pred.extend(out.argmax(1).cpu().numpy())
            all_true.extend(y_b.numpy())
    acc = accuracy_score(all_true, all_pred)
    f1  = f1_score(all_true, all_pred, average='macro')
    print(f"{name} — Test Accuracy: {acc:.4f}  |  Macro-F1: {f1:.4f}")
    return acc, f1, np.array(all_true), np.array(all_pred)


acc_a, f1_a, true_a, pred_a = evaluate_model(model_a, test_loader, 'Baseline CNN')
acc_b, f1_b, true_b, pred_b = evaluate_model(model_b, test_loader, 'Deep CNN    ')

print("\n=== Summary Comparison ===")
results_df = pd.DataFrame({
    'Model':     ['Baseline CNN', 'Deep CNN (BatchNorm + Dropout)'],
    'Params':    [total_params_a, total_params_b],
    'Test Acc':  [round(acc_a, 4), round(acc_b, 4)],
    'Macro-F1':  [round(f1_a, 4), round(f1_b, 4)],
    'Final Val Acc': [round(hist_a['val_acc'][-1], 4), round(hist_b['val_acc'][-1], 4)],
})
print(results_df.to_string(index=False))

In [ ]:
epochs_range = range(1, 31)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Loss
axes[0].plot(epochs_range, hist_a['train_loss'], 'b-',  lw=1.5, label='Baseline — Train')
axes[0].plot(epochs_range, hist_a['val_loss'],   'b--', lw=1.5, label='Baseline — Val')
axes[0].plot(epochs_range, hist_b['train_loss'], 'r-',  lw=1.5, label='Deep CNN — Train')
axes[0].plot(epochs_range, hist_b['val_loss'],   'r--', lw=1.5, label='Deep CNN — Val')
axes[0].set_title('Figure 1a — Cross-Entropy Loss Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, hist_a['train_acc'], 'b-',  lw=1.5, label='Baseline — Train')
axes[1].plot(epochs_range, hist_a['val_acc'],   'b--', lw=1.5, label='Baseline — Val')
axes[1].plot(epochs_range, hist_b['train_acc'], 'r-',  lw=1.5, label='Deep CNN — Train')
axes[1].plot(epochs_range, hist_b['val_acc'],   'r--', lw=1.5, label='Deep CNN — Val')
axes[1].set_title('Figure 1b — Accuracy Over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_ylim([0.6, 1.02])

plt.tight_layout()
plt.savefig('fig1_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print("Figure 1 interpretation: The Baseline CNN shows a widening train/val gap after epoch 15,"
      " indicating mild overfitting (train accuracy reaches 100% while val plateaus at ~98%)."
      " The Deep CNN converges faster and shows tighter train/val tracking due to Dropout and BatchNorm.")

In [ ]:
cm = confusion_matrix(true_b, pred_b)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=digits.target_names,
            yticklabels=digits.target_names, ax=ax)
ax.set_title('Figure 2 — Confusion Matrix: Deep CNN (Test Set, n=360)')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('fig2_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

# Identify error cases
errors = np.where(true_b != pred_b)[0]
print(f"\nTotal misclassifications: {len(errors)} / {len(true_b)} ({len(errors)/len(true_b)*100:.1f}%)")
for e in errors[:5]:
    print(f"  Sample {e}: True={true_b[e]}, Predicted={pred_b[e]}")

In [ ]:
# Show 20 test samples: all correct/error cases highlighted
model_b.eval()
X_test_t = torch.tensor(X_test[:20]).to(device)
with torch.no_grad():
    preds_sample = model_b(X_test_t).argmax(1).cpu().numpy()

fig, axes = plt.subplots(2, 10, figsize=(14, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i, 0] * 16, cmap='gray_r', vmin=0, vmax=16)
    correct = preds_sample[i] == y_test[i]
    color = 'green' if correct else 'red'
    status = '✓' if correct else '✗'
    ax.set_title(f'P:{preds_sample[i]} T:{y_test[i]} {status}', fontsize=8, color=color)
    ax.axis('off')

fig.suptitle('Figure 3 — Deep CNN Predictions on 20 Test Samples (Green=Correct, Red=Incorrect)',
             fontsize=10)
plt.tight_layout()
plt.savefig('fig3_sample_predictions.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"In these 20 samples: {(preds_sample == y_test[:20]).sum()} correct, "
      f"{(preds_sample != y_test[:20]).sum()} incorrect")

In [ ]:
# Per-class classification report for both models
print("=== Per-Class Report: Baseline CNN ===")
print(classification_report(true_a, pred_a, target_names=[str(i) for i in range(10)]))

print("\n=== Per-Class Report: Deep CNN ===")
print(classification_report(true_b, pred_b, target_names=[str(i) for i in range(10)]))